# Venturijeva cijev — utjecaj geometrije na tlak i brzinu

**Poglavlje 9: Bernoullijeva jednadžba idealnog fluida**

Ovaj interaktivni prikaz nadopunjuje izvod Bernoullijeve jednadžbe za stacionarno strujanje nestlačivog fluida kroz vodoravnu cijev s lokalnim suženjem. Mijenjanjem promjera ulaza, promjera grla i ulazne brzine prati se međusobna ovisnost brzine i tlaka uzduž cijevi.

Vrijednosti polaznih parametara prilagođene su riješenom primjeru iz poglavlja, ali se mogu slobodno mijenjati radi uočavanja graničnih slučajeva.

## Cilj

U Venturijevoj cijevi suženje grla mijenja brzinu fluida prema jednadžbi kontinuiteta, a Bernoullijeva jednadžba povezuje tu promjenu s padom tlaka. Prikaz omogućuje:

1. mijenjanje promjera ulaza $D_1$, promjera grla $D_2$ i ulazne brzine $v_1$;
2. promatranje promjene brzine i tlaka duž cijevi;
3. uočavanje međusobne veze energetske linije (EGL) i hidrauličke linije (HGL).

## Pretpostavke modela

- nestlačivi fluid (voda gustoće $\rho = 998$ kg/m³);
- stacionarno strujanje;
- vodoravna cijev ($z_1 = z_2$);
- zanemarivi gubici energije ($h_w = 0$);
- jednodimenzijski profil brzina u svakom presjeku.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Iz jednadžbe kontinuiteta za nestlačivi fluid slijedi izlazna brzina u grlu:

$$v_2 = v_1\,\frac{A_1}{A_2} = v_1\left(\frac{D_1}{D_2}\right)^2.$$

Bernoullijeva jednadžba između ulaza i grla (vodoravna cijev, bez gubitaka) daje pad tlaka:

$$p_2 = p_1 + \frac{\rho}{2}\left(v_1^2 - v_2^2\right).$$

Energetska linija EGL u idealnom modelu ostaje konstantna duž cijevi. Hidraulička linija HGL pada upravo u grlu, i to za iznos brzinske visine $v^2/(2g)$ koji odgovara lokalnoj brzini fluida.

In [ ]:
# Fizikalne konstante
RHO = 998.0           # gustoća vode (kg/m^3)
G = 9.81              # gravitacijska konstanta (m/s^2)
P1_REF = 150_000.0    # referentni ulazni tlak (Pa)

def venturi(D1_mm, D2_mm, v1):
    """Izračun stanja u Venturijevoj cijevi između ulaza i grla."""
    D1 = D1_mm / 1000.0
    D2 = D2_mm / 1000.0
    A1 = np.pi * D1**2 / 4
    A2 = np.pi * D2**2 / 4
    v2 = v1 * A1 / A2
    p2 = P1_REF + 0.5 * RHO * (v1**2 - v2**2)
    return {'v2': v2, 'p2': p2, 'A1': A1, 'A2': A2, 'dp': P1_REF - p2}

## Interaktivni prikaz

Klizačima u nastavku biraju se promjeri ulaza i grla te ulazna brzina. Gornji graf prikazuje aksijalni presjek cijevi, a donji graf prikazuje tlak, energetsku liniju i hidrauličku liniju duž osi cijevi.

In [ ]:
def venturi_profil(D1_mm, D2_mm, v1):
    # Osna koordinata u metrima; cijev je duga 1 m radi preglednosti.
    x = np.linspace(0, 1.0, 400)
    D = np.full_like(x, D1_mm, dtype=float)

    # Konvergentni dio: 0.30 m do 0.40 m
    mask_konv = (x >= 0.30) & (x < 0.40)
    D[mask_konv] = D1_mm + (D2_mm - D1_mm) * (x[mask_konv] - 0.30) / 0.10

    # Grlo: 0.40 m do 0.60 m
    mask_grlo = (x >= 0.40) & (x < 0.60)
    D[mask_grlo] = D2_mm

    # Difuzor: 0.60 m do 0.75 m
    mask_dif = (x >= 0.60) & (x < 0.75)
    D[mask_dif] = D2_mm + (D1_mm - D2_mm) * (x[mask_dif] - 0.60) / 0.15

    A = np.pi * (D / 1000.0)**2 / 4
    A1 = A[0]
    v = v1 * A1 / A
    p = P1_REF + 0.5 * RHO * (v1**2 - v**2)

    egl = (p + 0.5 * RHO * v**2) / (RHO * G)
    hgl = p / (RHO * G)

    fig, (ax_geo, ax_p) = plt.subplots(
        2, 1, figsize=(9, 6.5),
        gridspec_kw={'height_ratios': [1, 2]}
    )

    # Gornji prikaz: geometrija cijevi
    ax_geo.fill_between(x, -D/2, D/2, color='#aed6f1', alpha=0.6)
    ax_geo.plot(x, D/2, color='#1565c0', lw=1.8)
    ax_geo.plot(x, -D/2, color='#1565c0', lw=1.8)
    ax_geo.axhline(0, color='gray', ls=':', lw=0.6)
    ax_geo.set_xlim(0, 1.0)
    ax_geo.set_ylim(-D1_mm * 0.7, D1_mm * 0.7)
    ax_geo.set_ylabel('polumjer (mm)')
    ax_geo.set_title(
        f'Venturijeva cijev   |   $D_1$ = {D1_mm:.0f} mm,  '
        f'$D_2$ = {D2_mm:.0f} mm,  $v_1$ = {v1:.2f} m/s'
    )
    ax_geo.set_xticks([])

    # Donji prikaz: tlak i energijske linije
    ax_p.plot(x, p / 1000, color='#c62828', lw=2.2, label='tlak $p$ (kPa)')
    ax_p.set_xlabel('osna koordinata (m)')
    ax_p.set_ylabel('tlak (kPa)', color='#c62828')
    ax_p.tick_params(axis='y', labelcolor='#c62828')
    ax_p.grid(ls=':', alpha=0.5)

    ax_e = ax_p.twinx()
    ax_e.plot(x, egl, color='#2e7d32', lw=1.6, label='EGL (m)')
    ax_e.plot(x, hgl, color='#1565c0', lw=1.6, ls='--', label='HGL (m)')
    ax_e.set_ylabel('energijska visina (m)', color='#2e7d32')
    ax_e.tick_params(axis='y', labelcolor='#2e7d32')

    linije_p, oznake_p = ax_p.get_legend_handles_labels()
    linije_e, oznake_e = ax_e.get_legend_handles_labels()
    ax_p.legend(
        linije_p + linije_e, oznake_p + oznake_e,
        loc='lower right', framealpha=0.92
    )

    rezultat = venturi(D1_mm, D2_mm, v1)
    sazetak = (
        f"v_2 = {rezultat['v2']:.2f} m/s\n"
        f"p_2 = {rezultat['p2']/1000:.1f} kPa\n"
        f"Δp = {rezultat['dp']/1000:.1f} kPa"
    )
    ax_p.text(
        0.02, 0.96, sazetak, transform=ax_p.transAxes, fontsize=9,
        va='top', family='monospace',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#888', alpha=0.92)
    )

    plt.tight_layout()
    plt.show()


interact(
    venturi_profil,
    D1_mm=FloatSlider(min=40, max=120, step=2, value=80,
                       description='$D_1$ (mm)',
                       layout=Layout(width='420px')),
    D2_mm=FloatSlider(min=10, max=60, step=2, value=30,
                       description='$D_2$ (mm)',
                       layout=Layout(width='420px')),
    v1=FloatSlider(min=0.5, max=5.0, step=0.1, value=1.5,
                    description='$v_1$ (m/s)',
                    layout=Layout(width='420px'))
);

## Pitanja za istraživanje

Sljedeća pitanja sugeriraju načine korištenja gornjeg prikaza za samostalno istraživanje:

1. **Granični slučaj suženja.** Što se događa s tlakom u grlu kada se $D_2$ smanjuje prema 10 mm uz konstantni $D_1$ i $v_1$? U kojem trenutku tlak postaje negativan i je li ta vrijednost fizikalno smislena za stvarno strujanje vode?

2. **Omjer brzina i pad tlaka.** Za $v_1 = 2{,}0$ m/s, koja kombinacija $D_1$ i $D_2$ daje brzinu u grlu od približno $8{,}0$ m/s? Postoji li više rješenja i ovisi li pad tlaka isključivo o omjeru $D_1/D_2$ ili i o pojedinačnim vrijednostima promjera?

3. **Ponašanje energijskih linija.** Zašto energetska linija EGL u idealnom modelu ostaje konstantna duž cijevi, dok hidraulička linija HGL pada u grlu? Koje bi pretpostavke modela morale biti prekršene da EGL ne bi ostala konstantna i u kojem poglavlju se taj prijelaz uvodi?

4. **Bezdimenzijska invarijantnost.** Bezdimenzijski omjer $D_1/D_2$ određuje omjer brzina prema jednadžbi kontinuiteta. Provjerava se uz različite parove $(D_1, D_2)$ koji daju isti omjer 4:1 (na primjer 80/20, 100/25, 120/30) — daju li svi isti pad tlaka pri istoj $v_1$?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira dvije temeljne ideje iz poglavlja 9:

- **Jednadžba kontinuiteta** $A_1 v_1 = A_2 v_2$ određuje koliko se brzina pojačava u grlu Venturijeve cijevi.
- **Bernoullijeva jednadžba** $\dfrac{p_1}{\rho g} + \dfrac{v_1^2}{2g} = \dfrac{p_2}{\rho g} + \dfrac{v_2^2}{2g}$ određuje pripadni pad tlaka.

Kada se model proširi gubicima (poglavlje 10), energetska linija EGL više ne ostaje konstantna nego se postupno spušta duž cijevi, a hidraulička linija HGL prati novi tijek. U poglavlju 13 isti se princip primjenjuje na cijele cjevovodne sustave gdje se Venturijeva cijev koristi kao mjerni element protoka.